In [1]:
import pandas as pd

# Set display options for easier debugging
pd.set_option('display.max_columns', None)

# Load only needed columns to save memory
basics = pd.read_csv("C:\\Users\\bfire\\Desktop\\CSUSpring2025\\Data Mining\\Final Project\\Dataset\\title.basics.tsv", sep="\t", na_values="\\N", low_memory=False)
ratings = pd.read_csv("C:\\Users\\bfire\\Desktop\\CSUSpring2025\\Data Mining\\Final Project\\Dataset\\title.ratings.tsv", sep="\t", na_values="\\N")
crew = pd.read_csv("C:\\Users\\bfire\\Desktop\\CSUSpring2025\\Data Mining\\Final Project\\Dataset\\title.crew.tsv", sep="\t", na_values="\\N")
principals = pd.read_csv("C:\\Users\\bfire\\Desktop\\CSUSpring2025\\Data Mining\\Final Project\\Dataset\\title.principals.tsv", sep="\t", na_values="\\N")
names = pd.read_csv("C:\\Users\\bfire\\Desktop\\CSUSpring2025\\Data Mining\\Final Project\\Dataset\\name.basics.tsv", sep="\t", na_values="\\N")


In [14]:
# Filter for feature films only, not shorts, videos, etc.
movies = basics[
    (basics['titleType'] == 'movie') &
    (basics['startYear'].notna()) &
    (basics['startYear'].astype(str).str.isnumeric())
]

# Convert startYear to numeric
movies['startYear'] = movies['startYear'].astype(int)

# Keep movies from year 2000 onwards
movies = movies[movies['startYear'] >= 2000]


In [16]:
# Merge with ratings data
movies_with_ratings = pd.merge(movies, ratings, on='tconst')

# Filter for at least 1000 votes to reduce noise
movies_with_ratings = movies_with_ratings[movies_with_ratings['numVotes'] >= 1000]


In [18]:
# Merge crew data to add director and writer info
movies_with_crew = pd.merge(movies_with_ratings, crew, on='tconst', how='left')


In [28]:
# Join principals (for main cast and crew info)
top_principals = principals[principals['category'].isin(['actor', 'actress', 'director'])]

# Keep only the first 3 credited people per movie (optional for simplifying)
top_principals = top_principals[top_principals['ordering'] <= 3]

# Merge principals with names to get actor/actress names
top_principals_named = pd.merge(top_principals, names, on='nconst', how='left')

# Merge with movie data
merged_with_cast = pd.merge(movies_with_crew, top_principals_named, on='tconst', how='left')


In [30]:
movies_with_crew.to_csv("cleaned_imdb_movies.csv", index=False)

In [3]:
basics = pd.read_csv("C:\\Users\\bfire\\Desktop\\CSUSpring2025\\Data Mining\\Final Project\\Dataset\\title.basics.tsv", sep="\t", na_values="\\N",
                     usecols=['tconst', 'titleType', 'primaryTitle', 'startYear', 'isAdult', 'runtimeMinutes', 'genres'])
ratings = pd.read_csv("C:\\Users\\bfire\\Desktop\\CSUSpring2025\\Data Mining\\Final Project\\Dataset\\title.ratings.tsv", sep="\t", na_values="\\N")

crew = pd.read_csv("C:\\Users\\bfire\\Desktop\\CSUSpring2025\\Data Mining\\Final Project\\Dataset\\title.crew.tsv", sep="\t", na_values="\\N", usecols=['tconst', 'directors'])

basics['startYear'] = pd.to_numeric(basics['startYear'], errors='coerce')
basics['isAdult'] = pd.to_numeric(basics['isAdult'], errors='coerce')

# Now filter
movies = basics[
    (basics['titleType'] == 'movie') &
    (basics['startYear'] >= 2000)
]
print("After loading basics:", basics.shape)

movies['startYear'] = movies['startYear'].astype(int)
movies = movies[movies['startYear'] >= 2000]
print("After filtering movies:", movies.shape)

# Merge with ratings and filter again
movies_rated = pd.merge(movies, ratings, on='tconst')
#movies_rated = movies_rated[movies_rated['numVotes'] >= 1000]
print("After merging with ratings:", movies_rated.shape)

# Merge with director info
final = pd.merge(movies_rated, crew, on='tconst', how='left')
print("After merging with crew:", final.shape)

# Export a sample to CSV
final.sample(100000).to_csv("sample_imdb_movies_with_adult.csv", index=False)


C:\Users\bfire\AppData\Local\Temp\ipykernel_32332\1556856301.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  basics = pd.read_csv("C:\\Users\\bfire\\Desktop\\CSUSpring2025\\Data Mining\\Final Project\\Dataset\\title.basics.tsv", sep="\t", na_values="\\N",


After loading basics: (11590835, 7)
After filtering movies: (350555, 7)


C:\Users\bfire\AppData\Local\Temp\ipykernel_32332\1556856301.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies['startYear'] = movies['startYear'].astype(int)


After merging with ratings: (194234, 9)
After merging with crew: (194234, 10)
